In [2]:
import pandas as pd
import numpy as np
import os

# Raw 데이터 불러오기
train = pd.read_pickle(
    "../data/processed/mercari_eda_final.pkl")

train.head()

,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description
0,0,MLB Cincinnati Reds T Shirt Size XL,3,Men/Tops/T-shirts,NaN,10.0,1,No description yet
1,1,Razer BlackWidow Chroma Keyboard,3,Electronics/Computers & Tablets/Components & P...,Razer,52.0,0,This keyboard is in great condition and works ...
2,2,AVA-VIV Blouse,1,Women/Tops & Blouses/Blouse,Target,10.0,1,Adorable top with a hint of lace and a key hol...
3,3,Leather Horse Statues,1,Home/Home Décor/Home Décor Accents,NaN,35.0,1,New with tags. Leather horses. Retail for [rm]...
4,4,24K GOLD plated rose,1,Women/Jewelry/Necklaces,NaN,44.0,0,Complete with certificate of authenticity


In [3]:
train.shape

(1482535, 8)

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1482535 entries, 0 to 1482534
Data columns (total 8 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   train_id           1482535 non-null  int64  
 1   name               1482535 non-null  object 
 2   item_condition_id  1482535 non-null  int64  
 3   category_name      1476208 non-null  object 
 4   brand_name         849853 non-null   object 
 5   price              1482535 non-null  float64
 6   shipping           1482535 non-null  int64  
 7   item_description   1482529 non-null  object 
dtypes: float64(1), int64(3), object(4)
memory usage: 90.5+ MB


In [5]:
train.isnull().sum()

train_id                  0
name                      0
item_condition_id         0
category_name          6327
brand_name           632682
price                     0
shipping                  0
item_description          6
dtype: int64

In [6]:
df = train.copy()

In [7]:
df = df.drop(columns=["train_id"])

In [8]:
df.shape

(1482535, 7)

## 4.1 불필요한 변수 제거

`train_id`는 상품을 식별하기 위한 고유 식별자로 상품의 가격을 설명하는 특성을 포함하지 않는다.

따라서 모델이 식별자의 임의적인 수치 패턴을 학습하는 것을 방지하기 위해 `train_id`를 제거하였다.

원본 데이터는 유지하고 복사본인 `df`를 기준으로 이후 전처리를 수행한다.

In [9]:
(df["price"] == 0).sum()

np.int64(874)

In [10]:
df = df[df["price"] > 0].copy()

In [11]:
df.shape

(1481661, 7)

In [12]:
df["price"].describe()

count    1.481661e+06
mean     2.675329e+01
std      3.859198e+01
min      3.000000e+00
25%      1.000000e+01
50%      1.700000e+01
75%      2.900000e+01
max      2.009000e+03
Name: price, dtype: float64

### 4.2 비정상 가격 처리

EDA에서 `price`가 0인 상품이 874건 존재하는 것을 확인하였다.

본 분석의 목적은 상품의 실제 판매가격을 예측하는 것이므로, 가격이 0인 관측치는 정상적인 판매가격으로 해석하기 어렵다고 판단하였다.

이에 따라 `price > 0` 조건을 적용하여 해당 관측치를 제거하였다.

처리 결과 전체 1,482,535건에서 874건이 제외되어 최종 1,481,661건의 상품 데이터를 확보하였다.

가격의 최소값은 0에서 3으로 변경되었으며, 평균 26.75, 중앙값 17.00으로 나타났다.

In [13]:
# 브랜드 결측치
df["brand_name"] = df["brand_name"].fillna("Unknown")

# 카테고리 결측치
df["category_name"] = df["category_name"].fillna("Unknown")

# 상품 설명 결측치
df["item_description"] = df["item_description"].fillna("")

In [14]:
df.isnull().sum()

name                 0
item_condition_id    0
category_name        0
brand_name           0
price                0
shipping             0
item_description     0
dtype: int64

### 4.3 결측치 처리

결측치 분석 결과 `brand_name`에서 42.68%로 가장 높은 결측률이 나타났으며, `category_name`은 0.43%, `item_description`은 약 0.0004%의 결측률을 보였다.

`brand_name`은 결측 비율이 높기 때문에 해당 행을 제거할 경우 데이터 손실이 크게 발생한다. 또한 브랜드 정보 자체가 없는 상품도 실제 데이터의 일부일 수 있으므로 결측값을 별도의 범주인 `"Unknown"`으로 처리하였다.

`category_name` 역시 동일한 방식으로 `"Unknown"` 범주를 부여하였다.

`item_description`은 결측 비율이 극히 낮고 텍스트 분석 과정에서 빈 문자열로 처리할 수 있으므로 결측값을 빈 문자열(`""`)로 대체하였다.

처리 후 모든 변수에서 결측값이 존재하지 않음을 확인하였다.

In [15]:
category_split = df["category_name"].str.split("/", expand=True)

df["category_1"] = category_split[0]
df["category_2"] = category_split[1]
df["category_3"] = category_split[2]

In [16]:
category_split.head()

,0,1,2,3,4
0,Men,Tops,T-shirts,None,None
1,Electronics,Computers & Tablets,Components & Parts,None,None
2,Women,Tops & Blouses,Blouse,None,None
3,Home,Home Décor,Home Décor Accents,None,None
4,Women,Jewelry,Necklaces,None,None


In [17]:
df[
    ["category_1", "category_2", "category_3"]
] = df[
    ["category_1", "category_2", "category_3"]
].fillna("Unknown")

In [18]:
df[
    [
        "category_name",
        "category_1",
        "category_2",
        "category_3"
    ]
].head()

,category_name,category_1,category_2,category_3
0,Men/Tops/T-shirts,Men,Tops,T-shirts
1,Electronics/Computers & Tablets/Components & P...,Electronics,Computers & Tablets,Components & Parts
2,Women/Tops & Blouses/Blouse,Women,Tops & Blouses,Blouse
3,Home/Home Décor/Home Décor Accents,Home,Home Décor,Home Décor Accents
4,Women/Jewelry/Necklaces,Women,Jewelry,Necklaces


### 4.4.1 카테고리 계층 분리

`category_name`은 `/`를 기준으로 계층적인 상품 분류 정보를 포함하고 있다.

기존의 복합 문자열을 그대로 하나의 범주형 변수로 사용하는 대신, 상품 카테고리를 대분류(`category_1`), 중분류(`category_2`), 소분류(`category_3`)로 분리하였다.

이를 통해 상품의 계층적 특성을 유지하면서 각 수준의 카테고리가 가격에 미치는 영향을 독립적으로 분석할 수 있도록 전처리하였다.

In [19]:
brand_counts = (
    df["brand_name"]
    .value_counts()
)

brand_counts.head(20)

brand_name
Unknown              632336
PINK                  54072
Nike                  54006
Victoria's Secret     48011
LuLaRoe               30995
Apple                 17314
FOREVER 21            15178
Nintendo              14998
Lululemon             14550
Michael Kors          13916
American Eagle        13245
Rae Dunn              12300
Sephora               12164
Coach                 10458
Disney                10352
Bath & Body Works     10350
Adidas                10195
Funko                  9233
Under Armour           8458
Sony                   7992
Name: count, dtype: int64

In [20]:
brand_counts.describe()

count      4808.000000
mean        308.165765
std        9256.589557
min           1.000000
25%           1.000000
50%           4.000000
75%          23.000000
max      632336.000000
Name: count, dtype: float64

In [21]:
(brand_counts == 1).sum()

np.int64(1241)

In [22]:
# 기준을 최소 10개 상품으로 잡고, 그보다 적은 브랜드를 other로 묶음
brand_counts = df["brand_name"].value_counts()

# 10개 미만 등장 브랜드 → Other
rare_brands = brand_counts[brand_counts < 10].index

df["brand_name"] = df["brand_name"].replace(
    rare_brands,
    "Other"
)

In [23]:
df["brand_name"].nunique()

1754

In [24]:
df["brand_name"].value_counts().head(20)

brand_name
Unknown              632336
PINK                  54072
Nike                  54006
Victoria's Secret     48011
LuLaRoe               30995
Apple                 17314
FOREVER 21            15178
Nintendo              14998
Lululemon             14550
Michael Kors          13916
American Eagle        13245
Rae Dunn              12300
Sephora               12164
Coach                 10458
Disney                10352
Bath & Body Works     10350
Adidas                10195
Funko                  9233
Other                  8614
Under Armour           8458
Name: count, dtype: int64

### 4.4.2 브랜드 희소성 처리

`brand_name`은 4,808개의 고유 브랜드로 구성되어 있으며, 브랜드별 관측 빈도에 큰 편차가 존재하였다.

특히 중앙값은 4개 상품에 불과한 반면, 가장 많은 브랜드는 632,336개의 상품을 보유하고 있었다. 또한 상품이 1개만 존재하는 브랜드가 1,241개로 확인되어 브랜드 변수에 강한 희소성이 존재함을 확인하였다.

이러한 희소 브랜드를 모두 개별 범주로 유지할 경우 범주형 인코딩 과정에서 차원이 크게 증가하고, 관측치가 적은 범주에 대한 과적합 가능성이 높아질 수 있다.

따라서 본 분석에서는 상품 수가 10개 미만인 희소 브랜드를 `Other` 범주로 통합하여 범주 수를 축소하고 모델의 안정성을 높이고자 하였다.

해당 기준은 데이터의 빈도 분포를 바탕으로 설정한 전처리 기준이며, 이후 모델 성능 비교를 통해 적절성을 추가적으로 검증한다.

In [25]:
# 상품명, 상품설명 길이 확인
df["name_length"] = df["name"].str.len()

df["description_length"] = df["item_description"].str.len()

In [26]:
# 통계 확인
df[
    ["name_length", "description_length"]
].describe()

,name_length,description_length
count,1.481661e+06,1.481661e+06
mean,2.578670e+01,1.457145e+02
std,9.164531e+00,1.744462e+02
min,1.000000e+00,0.000000e+00
25%,1.900000e+01,4.000000e+01
50%,2.600000e+01,8.600000e+01
75%,3.400000e+01,1.740000e+02
max,4.300000e+01,1.046000e+03


### 4.5.1 텍스트 변수 탐색 및 파생변수 생성

Mercari 데이터의 `name`과 `item_description`은 상품의 특성을 직접적으로 포함하는 텍스트 변수이므로 가격 예측에서 정보 손실을 방지하기 위해 제거하지 않았다.

텍스트의 구조적 특성을 수치화하기 위해 상품명과 상품 설명의 문자 길이를 각각 `name_length`와 `description_length`로 생성하였다.

상품명의 평균 길이는 25.79자, 상품 설명의 평균 길이는 145.71자로 나타났다. 특히 상품 설명은 표준편차가 174.45로 평균보다 크고 최대 1,046자까지 나타나 상품별 설명 길이의 편차가 큰 것으로 확인되었다.

따라서 텍스트 자체에서 추출되는 단어 정보뿐만 아니라 텍스트 길이 역시 가격 예측에 활용할 수 있는 수치형 특성으로 판단하였다.

원문 텍스트는 이후 TF-IDF 기반 특성으로 변환하여 머신러닝 모델에 활용한다.

In [27]:
# 상품명
df["name"] = (
    df["name"]
    .fillna("")
    .str.strip()
)

# 상품 설명
df["item_description"] = (
    df["item_description"]
    .fillna("")
    .str.strip()
)

In [28]:
df["text"] = (
    df["name"].fillna("") + " " +
    df["item_description"].fillna("")
)

df["text"].head()

0    MLB Cincinnati Reds T Shirt Size XL No descrip...
1    Razer BlackWidow Chroma Keyboard This keyboard...
2    AVA-VIV Blouse Adorable top with a hint of lac...
3    Leather Horse Statues New with tags. Leather h...
4    24K GOLD plated rose Complete with certificate...
Name: text, dtype: object

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [30]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True
)

In [31]:
# TF-IDF 적용
# 10만 건 샘플
text_sample = df["text"].sample(
    100000,
    random_state=42
)

# TF-IDF 변환
tfidf_sample = tfidf.fit_transform(text_sample)

tfidf_sample.shape

(100000, 50000)

### 4.5.4 TF-IDF 기반 텍스트 특성 변환

상품명과 상품 설명은 가격에 영향을 줄 수 있는 핵심 텍스트 정보이므로 TF-IDF(Term Frequency-Inverse Document Frequency)를 활용하여 수치형 특성으로 변환하였다.

단순 단어 빈도만 사용할 경우 모든 상품에 자주 등장하는 단어의 영향이 과도하게 커질 수 있으므로, 특정 문서에 자주 등장하면서 전체 문서에서는 상대적으로 드문 단어에 높은 가중치를 부여하는 TF-IDF 방식을 적용하였다.

또한 단일 단어뿐만 아니라 연속된 두 단어의 조합을 고려하기 위해 unigram과 bigram을 함께 사용하였다.

전체 데이터에 적용하기 전 100,000건의 샘플을 이용하여 TF-IDF 변환을 테스트한 결과, 50,000개의 텍스트 특성이 정상적으로 생성되는 것을 확인하였다.

대규모 텍스트 데이터를 고려하여 희소행렬(sparse matrix) 기반으로 처리하였다.

In [32]:
tfidf_all = tfidf.transform(df["text"])

tfidf_all.shape

(1481661, 50000)

### 4.5.4 TF-IDF 기반 텍스트 특성 변환 결과

전체 1,481,661개의 상품에 대해 TF-IDF 변환을 수행한 결과,
상품별 최대 50,000개의 텍스트 특성이 생성되었다.

TF-IDF 결과는 대규모 텍스트 데이터의 고차원 특성을 효율적으로 처리하기 위해 희소행렬(sparse matrix) 형태로 유지하였다.

이를 통해 상품명과 상품 설명에 포함된 단어 및 단어 조합 정보를 가격 예측 모델의 입력 특성으로 활용할 수 있도록 변환하였다.

In [33]:
df["log_price"] = np.log1p(df["price"])


In [34]:
df["log_price"].describe()


count    1.481661e+06
mean     2.980816e+00
std      7.459274e-01
min      1.386294e+00
25%      2.397895e+00
50%      2.890372e+00
75%      3.401197e+00
max      7.605890e+00
Name: log_price, dtype: float64

In [35]:
df["log_price"].skew()
# 왜도

np.float64(0.6990683732394374)

## 4.6 Target 전처리

가격(`price`) 변수는 오른쪽으로 강하게 치우친 분포를 보였으며, 원본 가격의 왜도는 11.39로 나타났다.

가격 분포의 극단적인 왜도를 완화하고 고가 상품의 영향력을 줄이기 위해 `log1p(price)` 변환을 적용하였다.

로그 변환된 가격을 `log_price`라는 새로운 변수로 생성하여 모델의 Target으로 활용하고, 원본 `price`는 예측 결과를 실제 가격 단위로 복원하기 위해 유지하였다.

변환된 Target은 다음과 같이 정의하였다.

`log_price = log(1 + price)`

### 4.4.2 브랜드 희소성 처리

`brand_name`은 4,808개의 고유 브랜드로 구성되어 있으며, 브랜드별 관측 빈도에 큰 편차가 존재하였다.

특히 중앙값은 4개 상품에 불과한 반면, 가장 많은 브랜드는 632,336개의 상품을 보유하고 있었다. 또한 상품이 1개만 존재하는 브랜드가 1,241개로 확인되어 브랜드 변수에 강한 희소성이 존재함을 확인하였다.

이러한 희소 브랜드를 모두 개별 범주로 유지할 경우 범주형 인코딩 과정에서 차원이 크게 증가하고, 관측치가 적은 범주에 대한 과적합 가능성이 높아질 수 있다.

따라서 본 분석에서는 상품 수가 10개 미만인 희소 브랜드를 `Other` 범주로 통합하여 범주 수를 축소하고 모델의 안정성을 높이고자 하였다.

해당 기준은 데이터의 빈도 분포를 바탕으로 설정한 전처리 기준이며, 이후 모델 성능 비교를 통해 적절성을 추가적으로 검증한다.

In [36]:
# 전처리 최종점검

In [37]:
df.shape

(1481661, 14)

In [38]:
df.isnull().sum()

name                  0
item_condition_id     0
category_name         0
brand_name            0
price                 0
shipping              0
item_description      0
category_1            0
category_2            0
category_3            0
name_length           0
description_length    0
text                  0
log_price             0
dtype: int64

In [39]:
df.dtypes

name                   object
item_condition_id       int64
category_name          object
brand_name             object
price                 float64
shipping                int64
item_description       object
category_1             object
category_2             object
category_3             object
name_length             int64
description_length      int64
text                   object
log_price             float64
dtype: object

In [40]:
df.duplicated().sum()

np.int64(50)

In [41]:
# 중복 제거
df = df.drop_duplicates().copy()

In [42]:
# 제거 확인
df.duplicated().sum()

np.int64(0)

In [43]:
df.shape

(1481611, 14)

### 4.7.3 중복 데이터 처리

전처리 데이터에서 완전히 동일한 행이 50건 존재하는 것을 확인하였다.

동일한 관측치가 반복될 경우 특정 상품 정보가 모델 학습 과정에서 불필요하게 반복 반영될 수 있으므로 중복 행을 제거하였다.

중복 제거 후 데이터는 총 1,481,611건으로 감소하였으며, 이후 중복 여부를 재확인하여 중복 데이터가 제거되었음을 확인하였다.

In [44]:
# 타겟 확인
df[["price", "log_price"]].describe()

,price,log_price
count,1.481611e+06,1.481611e+06
mean,2.675383e+01,2.980838e+00
std,3.859250e+01,7.459255e-01
min,3.000000e+00,1.386294e+00
25%,1.000000e+01,2.397895e+00
50%,1.700000e+01,2.890372e+00
75%,2.900000e+01,3.401197e+00
max,2.009000e+03,7.605890e+00


### 4.7.4 Target 변수 점검

최종 전처리 데이터의 원본 가격(`price`)과 로그 변환 가격(`log_price`)의 분포를 확인하였다.

두 변수 모두 1,481,611개의 동일한 관측치를 가지고 있으며, `price`는 3~2,009 범위, `log_price`는 1.386~7.606 범위로 나타났다.

로그 변환을 통해 가격의 스케일을 축소하고 고가 상품의 영향력을 완화한 상태이며, 이후 머신러닝 모델의 Target으로 `log_price`를 사용한다.

원본 `price`는 모델 예측 결과를 실제 가격 단위로 복원하기 위해 별도로 유지한다.

In [45]:
df.columns.tolist()

['name',
 'item_condition_id',
 'category_name',
 'brand_name',
 'price',
 'shipping',
 'item_description',
 'category_1',
 'category_2',
 'category_3',
 'name_length',
 'description_length',
 'text',
 'log_price']

In [46]:
df.shape

(1481611, 14)

In [47]:
df.to_pickle("../data/processed/mercari_preprocessed.pkl")

print("저장 완료")
print("데이터 크기:", df.shape)

저장 완료
데이터 크기: (1481611, 14)


In [48]:
pd.to_pickle(
    tfidf_all,
    "../data/processed/tfidf_all.pkl"
)

print("저장 완료")
print("TF-IDF 크기:", tfidf_all.shape)

저장 완료
TF-IDF 크기: (1481661, 50000)


## 4.8 전처리 결과 저장

최종 전처리가 완료된 데이터를 이후 모델링 단계에서 재사용할 수 있도록 별도의 `processed` 폴더에 저장하였다.

일반적인 구조화 변수와 Target은 Pickle 형식으로 저장하고, 고차원 텍스트 데이터인 TF-IDF 결과는 희소행렬 형태로 저장하였다. 또한 동일한 TF-IDF 변환을 새로운 데이터에도 적용할 수 있도록 학습된 벡터라이저를 별도로 저장하였다.

이를 통해 이후 모델링 단계에서는 원본 데이터의 반복적인 전처리를 방지하고, 동일한 전처리 기준을 일관되게 적용할 수 있도록 구성하였다.